In [2]:


import sys
import subprocess
import importlib

def ensure_packages(packages):
    missing = []
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except Exception:
            missing.append(pkg)
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("All packages already installed:", packages)

ensure_packages(["selenium", "webdriver_manager", "beautifulsoup4", "pandas"])

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
import time, re
from bs4 import BeautifulSoup
import pandas as pd


def create_driver(headless=False, disable_images=True, page_load_strategy="eager"):
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
  
    if headless:
        options.add_argument("--headless=new")
        options.add_argument("--window-size=1920,1080")
    
    options.add_experimental_option("excludeSwitches", ["enable-logging"])
   
    options.page_load_strategy = page_load_strategy
    
    if disable_images:
        prefs = {
            "profile.managed_default_content_settings.images": 2,
            "profile.default_content_setting_values.notifications": 2
        }
        options.add_experimental_option("prefs", prefs)
   
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.implicitly_wait(1)  
    return driver

def make_waits(driver, short=5, long=15):
    return WebDriverWait(driver, short), WebDriverWait(driver, long)

def click_reviews_tab(driver, wait_short, wait_long):
    try:
        elem = wait_short.until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(normalize-space(.), 'REVIEWS', 'reviews'), 'review')]"))
        )
        driver.execute_script("arguments[0].click();", elem)
        try:
            wait_short.until(EC.presence_of_element_located((By.CSS_SELECTOR, "li.list-item")))
        except TimeoutException:
            try:
                wait_long.until(EC.presence_of_element_located((By.CSS_SELECTOR, "li.list-item")))
            except TimeoutException:
                time.sleep(0.5)
        return True
    except Exception:
        return False

def repeatedly_click_load_more(driver, wait_short, max_retries=100):
    retries = 0
    prev_count = 0

    while retries < max_retries:
        reviews = driver.find_elements(By.CSS_SELECTOR, "li.list-item, div#reviewContainer div.bv-content-item")
        curr_count = len(reviews)

        buttons = driver.find_elements(
            By.XPATH,
            "//a[contains(translate(., 'LOAD MOREVIEW ALLREVIEWS', 'load moreview allreviews'), 'load more') or "
            "contains(translate(., 'VIEW ALL', 'view all'), 'view all') or "
            "contains(translate(., 'VIEW ALL REVIEWS', 'view all reviews'), 'view all reviews')] | "
            "//button[contains(translate(., 'LOAD MOREVIEW ALLREVIEWS', 'load moreview allreviews'), 'load more') or "
            "contains(., 'View All')]"
        )

        if not buttons:
            print("No more load buttons found")
            break

        btn = buttons[0]
        if not (btn.is_displayed() and btn.is_enabled()):
            print("Button not clickable anymore")
            break

        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            driver.execute_script("arguments[0].click();", btn)
        except WebDriverException:
            print("JS click failed, trying fallback")
            driver.execute_script("""
                const el = Array.from(document.querySelectorAll('a,button'))
                    .find(x => /(view\\s*all\\s*reviews|view\\s*all|load\\s*more)/i.test(x.textContent));
                if(el) el.click();
            """)

        try:
            wait_short.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "li.list-item, div#reviewContainer div.bv-content-item")) > curr_count)
            prev_count = curr_count
            retries = 0  
        except TimeoutException:
            retries += 1
            print("No new reviews loaded, retry:", retries)

    print("Final review count:", len(driver.find_elements(By.CSS_SELECTOR, "li.list-item, div#reviewContainer div.bv-content-item")))

def extract_reviews_from_soup(soup):
    reviews = []
    for li in soup.select("li.list-item"):
        rating_div = li.select_one("div.cp-rating.user-review-rating")
        rating_text = rating_div.get_text(" ", strip=True) if rating_div else ""
        rating_match = re.search(r"\(?\b(\d+(?:\.\d+)?)\b\)?", rating_text)
        rating = rating_match.group(1) if rating_match else ""
        review_div = li.select_one("div.desc.user-reviews-descr")
        if not review_div:
            review_div = li.select_one(".review-text, .review-body, [itemprop='reviewBody']")
        if review_div:
            raw_text = review_div.get_text(" ", strip=True)
            review_text = re.sub(r"\s+", " ", raw_text)
        else:
            review_text = ""
        if review_text:
            reviews.append({"rating": rating, "review": review_text})
    return reviews

def try_extract_from_iframes(driver):
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    for idx, iframe in enumerate(iframes):
        try:
            driver.switch_to.frame(iframe)
            inner = driver.page_source
            if "list-item" in inner or "review" in inner.lower():
                soup = BeautifulSoup(inner, "html.parser")
                items = soup.select("li.list-item")
                if items:
                    driver.switch_to.default_content()
                    return soup
            driver.switch_to.default_content()
        except Exception:
            try:
                driver.switch_to.default_content()
            except Exception:
                pass
            continue
    return None

def scrape_croma_reviews(product_url, output_csv="croma_reviews.csv", headless=True):
    driver = create_driver(headless=headless)
    wait_short, wait_long = make_waits(driver, short=5, long=15)
    try:
        driver.get(product_url)

        try:
            wait_short.until(lambda d: d.execute_script("return document.readyState") in ("interactive", "complete"))
        except TimeoutException:
            time.sleep(0.5)

        clicked = click_reviews_tab(driver, wait_short, wait_long)
        if not clicked:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
            time.sleep(0.4)
            clicked = click_reviews_tab(driver, wait_short, wait_long)

        repeatedly_click_load_more(driver, wait_short, max_retries=12)

        soup = try_extract_from_iframes(driver)
        if soup is None:
            soup = BeautifulSoup(driver.page_source, "html.parser")

        reviews = extract_reviews_from_soup(soup)

        if len(reviews) == 0:
            debug_html = driver.page_source
            with open("debug_croma_page.html", "w", encoding="utf-8") as f:
                f.write(debug_html)
            print("⚠ No reviews found. Saved page to debug_croma_page.html for inspection.")

            soup_all = BeautifulSoup(debug_html, "html.parser")
            fallback_items = [tag for tag in soup_all.find_all(True) if "review" in " ".join(tag.get("class", [])).lower()]
            if fallback_items:
                for tag in fallback_items[:20]:
                    txt = tag.get_text(" ", strip=True)
                    txt = re.sub(r"\s+", " ", txt)
                    if len(txt) > 20:
                        reviews.append({"rating": "", "review": txt})
                print(f"⚠ Fallback found {len(reviews)} possible review-like items (useful for debugging selectors).")

        df = pd.DataFrame(reviews)
        df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"Saved {len(df)} reviews to {output_csv}")
        return df

    finally:
        try:
            driver.quit()
        except Exception:
            pass

def url_link(product_url):
    return scrape_croma_reviews(product_url, output_csv=product_url[-6:] + ".csv", headless=True)

if __name__ == "__main__":
    pass

Installing missing packages: ['beautifulsoup4']


In [ ]:
url_link("https://www.croma.com/apple-iphone-13-128gb-starlight-white-/p/243460")

In [ ]:
inp=url_link(input('Enter link of any product of croma'))
inp

Enter link of any product of croma https://www.croma.com/apple-iphone-13-128gb-starlight-white-/p/243460
